# Lab 06 — 05 Governance: GRANT, RLS and CLS

This notebook demonstrates Unity Catalog governance on the Synthea Gold model.

- **GRANTs**: object permissions
- **RLS**: organization-based access on `fact_encounters`
- **CLS**: masking synthetic patient identifiers in `dim_patient`

The demo temporarily restricts the invoking user, validates the restriction,
then restores full access for that user while leaving the policies attached.

## 1. Runtime parameters

In [0]:
dbutils.widgets.text("catalog", "dbr_dev", "01 Catalog")
dbutils.widgets.text("schema", "parvinbadalov", "02 Schema")
dbutils.widgets.text("volume_name", "lab06_gold_analytics", "03 External volume")
dbutils.widgets.dropdown("run_demo", "true", ["true", "false"], "04 Run demo")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume_name = dbutils.widgets.get("volume_name")
run_demo = dbutils.widgets.get("run_demo").lower() == "true"

print(f"Catalog  : {catalog}")
print(f"Schema   : {schema}")
print(f"Volume   : {volume_name}")
print(f"Run demo : {run_demo}")

## 2. Shared configuration

In [0]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

current_dir = Path.cwd()
lab_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.config import Lab06Config

config = Lab06Config(
    catalog=catalog,
    schema=schema,
    volume_name=volume_name,
)

user_org_access = config.table("lab06_user_organization_access")
privileged_users = config.table("lab06_patient_data_privileged_users")
org_filter_function = config.table("lab06_org_row_filter")
mask_function = config.table("lab06_mask_sensitive_string")

print(f"RLS target : {config.fact_encounters}")
print(f"CLS target : {config.dim_patient}")

## 3. Resolve the invoking user

`SESSION_USER()` is appropriate here because the policy must evaluate the
identity of the querying user. It is not used to construct storage paths.

In [0]:
session_user = spark.sql(
    "SELECT SESSION_USER() AS username"
).first()["username"]

def quote_principal(value: str) -> str:
    return "`" + value.replace("`", "``") + "`"

principal_sql = quote_principal(session_user)
escaped_user = session_user.replace("'", "''")

print(f"Session user: {session_user}")

## 4. Validate governance targets and capture baseline

In [0]:
for table_name in [config.fact_encounters, config.dim_patient]:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Missing governance target: {table_name}")

baseline_encounter_count = spark.table(config.fact_encounters).count()

demo_org_row = (
    spark.table(config.fact_encounters)
    .filter(F.col("organization_id").isNotNull())
    .groupBy("organization_id")
    .count()
    .orderBy(F.desc("count"))
    .first()
)

if demo_org_row is None:
    raise RuntimeError("No organization_id is available for the RLS demo.")

demo_organization_id = demo_org_row["organization_id"]
demo_organization_count = int(demo_org_row["count"])

print(f"Baseline rows          : {baseline_encounter_count:,}")
print(f"Demo organization      : {demo_organization_id}")
print(f"Demo organization rows : {demo_organization_count:,}")

## 5. Create governance mapping tables

In [0]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS {user_org_access} (
    username STRING NOT NULL,
    organization_id STRING NOT NULL,
    access_reason STRING,
    updated_at TIMESTAMP
)
USING DELTA
''')

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {privileged_users} (
    username STRING NOT NULL,
    access_reason STRING,
    updated_at TIMESTAMP
)
USING DELTA
''')

print("Governance mapping tables are ready.")

## 6. Create the organization row-filter function

The policy is default-deny:
- matching organization → visible
- `*` mapping → all organizations
- no mapping → no encounter rows

**Implementation note:** the UDF argument is named `row_organization_id` so it cannot be confused with the `organization_id` column in the mapping table.


In [0]:
spark.sql(f'''
CREATE OR REPLACE FUNCTION {org_filter_function} (
    row_organization_id STRING
)
RETURN EXISTS (
    SELECT 1
    FROM {user_org_access} a
    WHERE a.username = SESSION_USER()
      AND (
          a.organization_id = row_organization_id
          OR a.organization_id = '*'
      )
)
''')

print(f"Created: {org_filter_function}")

## 7. Prepare restricted RLS demo access

In [0]:
if run_demo:
    escaped_org = demo_organization_id.replace("'", "''")

    spark.sql(
        f"DELETE FROM {user_org_access} "
        f"WHERE username = '{escaped_user}'"
    )

    spark.sql(f'''
    INSERT INTO {user_org_access}
    VALUES (
        '{escaped_user}',
        '{escaped_org}',
        'Lab 06 restricted RLS demo',
        CURRENT_TIMESTAMP()
    )
    ''')

    display(
        spark.table(user_org_access)
        .filter(F.col("username") == session_user)
    )
else:
    print("Restricted RLS demo skipped.")

## 8. Apply RLS to `fact_encounters`

In [0]:
spark.sql(f'''
ALTER TABLE {config.fact_encounters}
SET ROW FILTER {org_filter_function}
ON (organization_id)
''')

print("RLS policy attached.")

## 9. Validate restricted RLS

In [0]:
if run_demo:
    restricted_encounter_count = spark.table(
        config.fact_encounters
    ).count()

    rls_restricted_status = (
        "PASS"
        if restricted_encounter_count == demo_organization_count
        else "FAIL"
    )

    display(
        spark.createDataFrame(
            [(
                baseline_encounter_count,
                demo_organization_count,
                restricted_encounter_count,
                rls_restricted_status,
            )],
            [
                "baseline_rows",
                "expected_restricted_rows",
                "actual_restricted_rows",
                "status",
            ],
        )
    )

    if rls_restricted_status == "FAIL":
        raise RuntimeError("Restricted RLS validation failed.")
else:
    rls_restricted_status = "SKIPPED"

## 10. Restore full RLS access for the invoking user

The row filter remains active. The current user receives a wildcard mapping so
normal development can continue. Unmapped users remain default-denied.

In [0]:
spark.sql(
    f"DELETE FROM {user_org_access} "
    f"WHERE username = '{escaped_user}'"
)

spark.sql(f'''
INSERT INTO {user_org_access}
VALUES (
    '{escaped_user}',
    '*',
    'Lab 06 full organization access',
    CURRENT_TIMESTAMP()
)
''')

restored_encounter_count = spark.table(config.fact_encounters).count()

rls_restored_status = (
    "PASS"
    if restored_encounter_count == baseline_encounter_count
    else "FAIL"
)

display(
    spark.createDataFrame(
        [(
            baseline_encounter_count,
            restored_encounter_count,
            rls_restored_status,
        )],
        ["baseline_rows", "restored_rows", "status"],
    )
)

if rls_restored_status == "FAIL":
    raise RuntimeError("RLS full-access restoration failed.")

## 11. Create the patient-data masking function

In [0]:
spark.sql(f'''
CREATE OR REPLACE FUNCTION {mask_function} (
    value STRING
)
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM {privileged_users} p
        WHERE p.username = SESSION_USER()
    )
    THEN value
    WHEN value IS NULL THEN NULL
    ELSE '***MASKED***'
END
''')

print(f"Created: {mask_function}")

## 12. Apply CLS masks to `dim_patient`

In [0]:
SENSITIVE_COLUMNS = [
    "ssn",
    "first_name",
    "last_name",
    "address",
]

for column_name in SENSITIVE_COLUMNS:
    spark.sql(f'''
    ALTER TABLE {config.dim_patient}
    ALTER COLUMN {column_name}
    SET MASK {mask_function}
    ''')

print("Masked columns: " + ", ".join(SENSITIVE_COLUMNS))

## 13. Validate masked CLS behavior

In [0]:
if run_demo:
    spark.sql(
        f"DELETE FROM {privileged_users} "
        f"WHERE username = '{escaped_user}'"
    )

    masked_sample = (
        spark.table(config.dim_patient)
        .select(
            "patient_id",
            "ssn",
            "first_name",
            "last_name",
            "address",
        )
        .limit(5)
    )

    display(masked_sample)

    unmasked_non_null_ssn = (
        masked_sample
        .filter(
            F.col("ssn").isNotNull()
            & (F.col("ssn") != "***MASKED***")
        )
        .count()
    )

    cls_masked_status = (
        "PASS" if unmasked_non_null_ssn == 0 else "FAIL"
    )

    if cls_masked_status == "FAIL":
        raise RuntimeError("CLS masked-value validation failed.")
else:
    cls_masked_status = "SKIPPED"

## 14. Restore privileged CLS access for the invoking user

Masks stay attached. The current user is added to the privileged mapping, while
unmapped users continue to receive masked values.

In [0]:
spark.sql(
    f"DELETE FROM {privileged_users} "
    f"WHERE username = '{escaped_user}'"
)

spark.sql(f'''
INSERT INTO {privileged_users}
VALUES (
    '{escaped_user}',
    'Lab 06 patient-data privileged access',
    CURRENT_TIMESTAMP()
)
''')

privileged_sample = (
    spark.table(config.dim_patient)
    .select(
        "patient_id",
        "ssn",
        "first_name",
        "last_name",
        "address",
    )
    .filter(F.col("ssn").isNotNull())
    .limit(5)
)

display(privileged_sample)

cls_restored_status = (
    "PASS"
    if privileged_sample
       .filter(F.col("ssn") == "***MASKED***")
       .count() == 0
    else "FAIL"
)

if cls_restored_status == "FAIL":
    raise RuntimeError("CLS restoration validation failed.")

## 15. Apply and verify GRANTs

For this personal lab run, the resolved session user is the GRANT principal.
In a shared environment, use account-level groups instead.

In [0]:
grant_statements = [
    (
        f"GRANT USE CATALOG ON CATALOG {catalog} "
        f"TO {principal_sql}"
    ),
    (
        f"GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} "
        f"TO {principal_sql}"
    ),
    (
        f"GRANT SELECT ON TABLE {config.fact_encounters} "
        f"TO {principal_sql}"
    ),
    (
        f"GRANT SELECT ON TABLE {config.dim_patient} "
        f"TO {principal_sql}"
    ),
]

for statement in grant_statements:
    spark.sql(statement)

display(
    spark.sql(
        f"SHOW GRANTS ON TABLE {config.fact_encounters}"
    )
)

print(f"GRANTs applied to: {session_user}")

## 16. Final governance validation

In [0]:
final_checks = [
    ("RLS restricted demo", rls_restricted_status),
    ("RLS full-access restoration", rls_restored_status),
    ("CLS masked demo", cls_masked_status),
    ("CLS privileged restoration", cls_restored_status),
]

final_validation_df = spark.createDataFrame(
    final_checks,
    ["governance_check", "status"],
)

display(final_validation_df)

failed_checks = [
    check_name
    for check_name, status in final_checks
    if status == "FAIL"
]

if failed_checks:
    raise RuntimeError(
        "Governance validation failed: "
        + ", ".join(failed_checks)
    )

## 17. Final policy state

```text
fact_encounters
  └── RLS attached
      ├── current user → "*" → all organizations
      └── unmapped user → no rows

dim_patient
  ├── ssn        → masked for non-privileged users
  ├── first_name → masked for non-privileged users
  ├── last_name  → masked for non-privileged users
  └── address    → masked for non-privileged users
```

The policies remain attached after the notebook completes.

**Next:** `lab06_06_alert_simulation`.

In [0]:
print("LAB 06 — GOVERNANCE COMPLETE")
print(f"RLS target : {config.fact_encounters}")
print(f"CLS target : {config.dim_patient}")
print(f"Principal  : {session_user}")
print("RLS and CLS remain enabled.")
print("Next: lab06_06_alert_simulation")

## Optional rollback

Do not run rollback during the normal Lab 06 flow.

The rollback SQL is documented in `sql/governance_policies.sql`.
Always remove a row filter or column mask from its table **before** dropping the
UDF referenced by that policy.